# Thư viện

In [11]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Gộp file

In [12]:
import pandas as pd
import json
import glob
import os

# 1. ĐƯỜNG DẪN THƯ MỤC CHỨA CÁC FILE JSON
folder_path = r"D:\PYTHON\Data mining\VN-IT-Job-Mining\data\raw\vieclam24h\2026-10-20"

# Tìm toàn bộ các file có đuôi .json trong thư mục này
json_files = glob.glob(os.path.join(folder_path, "*.json"))

# Khởi tạo list để chứa dữ liệu từ tất cả các file
raw_payloads = []

# 2. VÒNG LẶP ĐỌC VÀ GỘP DỮ LIỆU
print(f"Đang tiến hành đọc {len(json_files)} file JSON...")

for file in json_files:
    try:
        with open(file, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
            # Kịch bản A: File JSON là một đối tượng (Dictionary)
            if isinstance(data, dict):
                if 'raw_payload' in data:
                    payload = data['raw_payload']
                    
                    # ---> FIX IDENTIFIER ISSUE: Lấy metadata nhúng vào payload <---
                    metadata_block = data.get('metadata', {})
                    
                    if isinstance(payload, list):
                        # Nếu payload là list, nhúng metadata vào từng tin tuyển dụng bên trong
                        for item in payload:
                            if isinstance(item, dict):
                                item['metadata'] = metadata_block
                        raw_payloads.extend(payload)
                    else:
                        # Nếu payload là dict, tạo một key 'metadata' chứa thông tin định danh
                        payload['metadata'] = metadata_block
                        raw_payloads.append(payload)
                else:
                    raw_payloads.append(data)
                    
            # Kịch bản B: File JSON là một mảng (List chứa nhiều jobs)
            elif isinstance(data, list):
                for item in data:
                    if isinstance(item, dict) and 'raw_payload' in item:
                        payload = item['raw_payload']
                        metadata_block = item.get('metadata', {})
                        
                        if isinstance(payload, list):
                            for p in payload:
                                if isinstance(p, dict):
                                    p['metadata'] = metadata_block
                            raw_payloads.extend(payload)
                        else:
                            payload['metadata'] = metadata_block
                            raw_payloads.append(payload)
                    else:
                        raw_payloads.append(item)
                        
    except Exception as e:
        print(f"❌ Lỗi khi đọc file {file}: {e}")

# 3. CHUYỂN ĐỔI THÀNH DATAFRAME VÀ BÓC TÁCH NESTED JSON
# Bằng việc nhúng metadata vào payload, json_normalize sẽ tạo ra các cột:
# 'metadata_job_id', 'metadata_source_url' cùng với các cột của raw_payload
df_vieclam24h = pd.json_normalize(raw_payloads, sep='_')

# 4. KIỂM TRA VÀ IN KẾT QUẢ
print("\n✅ Đã gộp thành công toàn bộ dữ liệu Vieclam24h (Bao gồm cả Metadata Định danh)!")
print(f"Số lượng bản ghi (dòng): {df_vieclam24h.shape[0]}")
print(f"Số lượng trường dữ liệu (cột): {df_vieclam24h.shape[1]}")

print("\n--- IN RA 3 DÒNG ĐẦU TIÊN CỦA DATAFRAME ---")
pd.set_option('display.max_columns', None)  # Hiển thị toàn bộ cột
display(df_vieclam24h.head(3))

Đang tiến hành đọc 620 file JSON...

✅ Đã gộp thành công toàn bộ dữ liệu Vieclam24h (Bao gồm cả Metadata Định danh)!
Số lượng bản ghi (dòng): 1235
Số lượng trường dữ liệu (cột): 42

--- IN RA 3 DÒNG ĐẦU TIÊN CỦA DATAFRAME ---


,title,salary,deadline,location_summary,experience_summary,description,requirements,benefits,working_time,working_address,keywords,general_info_posted_date,general_info_level,general_info_hiring_quantity,general_info_working_type,general_info_experience,general_info_industries,general_info_workplaces,company_name,company_address,company_size,metadata_job_id,metadata_source,metadata_source_url,metadata_scraped_at,metadata_scraper_version,metadata_version,metadata_content_hash,metadata_canonical_hash,metadata_field_hashes_title,metadata_field_hashes_salary,metadata_field_hashes_deadline,metadata_field_hashes_location_summary,metadata_field_hashes_experience_summary,metadata_field_hashes_description,metadata_field_hashes_requirements,metadata_field_hashes_benefits,metadata_field_hashes_working_time,metadata_field_hashes_working_address,metadata_field_hashes_general_info,metadata_field_hashes_keywords,metadata_field_hashes_company
0,Nhân Viên Kinh Doanh - Tư Vấn Giải Pháp Lương ...,12 - 20 triệu,31/10/2026,Hà Nội,1 năm,"[Tìm kiếm, phát triển và mở rộng tệp khách hàn...",[Tốt nghiệp Cao đẳng/Đại học các ngành Kinh tế...,[Mức lương cơ bản:12.000.000 - 15.000.000 + ph...,,"Tầng 24, tòa MHDI, 68 Nguyễn Cơ Thạch, Mỹ Đình...","[Tư Vấn, nhân viên tư vấn visa, nhân viên tư v...",29/09/2026,Nhân viên,3,Toàn thời gian cố định,1 năm,Marketing / Bán hàng - Kinh doanh / IT Phần mềm,"[Hà Nội : Tầng 24, tòa MHDI, 68 Nguyễn Cơ Thạc...",Công Ty Cổ Phần Công Nghệ Ladipage Việt Nam,"Tầng 24, tòa MHDI, 68 Nguyễn Cơ Thạch, Mỹ Đình...",10 - 150 nhân viên,100160858,vieclam24h,https://vieclam24h.vn/marketing/nhan-vien-kinh...,2026-10-20T13:22:26.328038+00:00,2.0.0,1,a1d94fe8cf7c8dfb49c7466480eebfa2fd56ea8a1b85a8...,a1d94fe8cf7c8dfb49c7466480eebfa2fd56ea8a1b85a8...,6af712e0ac6454992a589bb55e60d326e56a23017d649c...,c4f2366c68fab768ac79562231cf6047dfe3c54882b506...,d9e8a0b6c1d526bb86793191f30bfc0441a9faefa9d051...,6ac69af836de8007fe7da283ce10821ab484ae1af98a30...,2c44e3b576e2d189d514341765d08189ee12f3204a3e6c...,23c1de3f61de3807d39b58476520017d2e9b0383f87b84...,f494628916c674edb673d249c70f5d2729eff1d330bcd8...,b501b7cf2870154e0be8e73023672a6420cf5f64955a1b...,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,8ff8eb81d5de5f2009a67c640bf25301ac3f002c9337be...,76933e4b4ad2bc75e2fa442ac64d0cd08f759885a5dcf1...,9266046b2b652f290dd93ef838ffb9e7bef81a3dfefa46...,5132d242ae65cad10cced54c82e3b41e5a704ec2f9fee9...
1,IT Helpdesk,8 - 12 triệu,31/10/2026,TP.HCM,2 năm,"[Cài đặt, xử lý các sự cố về máy tính, máy in ...",[Tốt nghiệp Cao đẳng/ Đại học chuyên ngành CNT...,"[Lương: Thỏa thuận tùy theo kinh nghiệm, Chính...",,"74/18 Trương Quốc Dung, Phường 10, Quận Phú Nh...","[it helpdesk, it support / it helpdesk, It, it...",29/09/2026,Nhân viên,1,Toàn thời gian cố định,2 năm,IT Phần cứng - Mạng,"[TP.HCM : 13C Hồ Biểu Chánh, Quận Phú Nhuận]",Công Ty TNHH TM Dược Phẩm Úc Châu,"74/18 Trương Quốc Dung, Phường 10, Quận Phú Nh...",10 - 150 nhân viên,200011696,vieclam24h,https://vieclam24h.vn/it-phan-cung-mang/it-hel...,2026-10-20T13:42:10.741317+00:00,2.0.0,1,7b06eea7f56d15a31e3bc7cf7f8262270a8d1ad0803503...,7b06eea7f56d15a31e3bc7cf7f8262270a8d1ad0803503...,dfa48b58549db64ea1c8ab00daf3b73689ec5efc796673...,61c0a6a087233e08c6eced01a088c6f5f04c7a7cb436ab...,d9e8a0b6c1d526bb86793191f30bfc0441a9faefa9d051...,f8b2201def938f296335572141bcac75bfe80b8867f106...,2da9cf09d3666c71d985c3879870885828da5f72c5f090...,1e97f7164e00c0bd8472d8212d24f69b90caa04d50c302...,f9bc8a7e69bcd10184dbd23362fffc6a444363ed65894e...,60f855b5219fd27f8ff4dfcc973964e55853562eea3223...,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,9a67c532b0f4dd4d6355652e8c52e60cb279ca3d791f32...,00341574817f42a236b3484e23bc4851c9a932559fcaa8...,2500f34630857e6214bbc4eb18578ce5004b34e17873e0...,d3e23c4516ef47098863755a2f10424ffff83f8e178744...
2,Full – Stack Java Developer (Onsite),20 - 25 triệu,12/11/2026,Hà Nội,2 năm,[- Phát triển hệ thống ebanking của ngân hàng ...,"[Kiến thức:, - Tốt nghiệp kỹ sư chuyên ngành C...",[- Thu n

# Trụ Cột 1: Độ Hoàn Thiện (Completeness & Missing Value Profiling)

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# ==========================================
# 1. HÀM QUÉT VÀ LÀM SẠCH NULL MARKERS ẨN
# ==========================================
def clean_hidden_nulls(df):
    """
    Nhận diện và chuyển đổi các giá trị khuyết thiếu dạng text hoặc mảng rỗng về chuẩn np.nan
    """
    df_cleaned = df.copy()
    
    # 1.1. Xử lý các null markers dạng chuỗi
    null_markers = [
        "", " ", "N/A", "n/a", "null", "Null", "NULL", "None", "none",
        "Chưa cập nhật", "Đang cập nhật", "Thỏa thuận", "Thoả thuận", "Thương lượng"
    ]
    
    for col in df_cleaned.select_dtypes(include=['object']):
        df_cleaned[col] = df_cleaned[col].apply(
            lambda x: np.nan if str(x).strip() in null_markers else x
        )
    
    # 1.2. Xử lý các list/dict rỗng trên toàn bộ dataframe
    df_cleaned = df_cleaned.map(
        lambda x: np.nan if isinstance(x, (list, dict)) and len(x) == 0 else x
    )
    
    return df_cleaned

# ==========================================
# 2. HÀM TRỰC QUAN HÓA: BAR CHART KHUYẾT THIẾU (TOÀN BỘ CỘT)
# ==========================================
def plot_completeness_bar_all(df, save_path):
    """
    Vẽ Bar Chart ngang thể hiện tỷ lệ khuyết thiếu của TẤT CẢ CÁC CỘT trong DataFrame.
    """
    # Tính tỷ lệ % null cho toàn bộ cột
    null_percentages = (df.isnull().sum() / len(df)) * 100
    
    # Chỉ lấy các cột bị khuyết thiếu (Tỷ lệ > 0)
    cols_with_nulls = null_percentages[null_percentages > 0]
    
    if cols_with_nulls.empty:
        print("✅ Dữ liệu hoàn hảo, không có cột nào bị khuyết thiếu!")
        return

    # Sắp xếp để thanh dài nhất (tỷ lệ thiếu cao nhất) nằm trên cùng
    missing_pct_sorted = cols_with_nulls.sort_values(ascending=True)

    # Tự động điều chỉnh chiều cao ảnh: mỗi cột chiếm 0.4 inch, tối thiểu là 8 inch vì số lượng cột lớn
    fig_height = max(8, len(missing_pct_sorted) * 0.4)
    plt.figure(figsize=(14, fig_height))
    
    # Vẽ biểu đồ thanh ngang
    bars = plt.barh(missing_pct_sorted.index, missing_pct_sorted.values, color="#e76f51", alpha=0.85)
    
    # Tùy chỉnh tiêu đề và trục
    plt.title("Tỷ Lệ Khuyết Thiếu Từng Trường (%) - Toàn Bộ Cột (Vieclam24h)", fontsize=15, pad=20, fontweight='bold')
    plt.xlabel("Tỷ lệ Null (%)", fontsize=12)
    plt.yticks(fontsize=10)
    
    # Mở rộng không gian trục X để text % không bị lẹm ra ngoài
    plt.xlim(0, max(missing_pct_sorted.values) + 15) 
    
    # In trực tiếp tỷ lệ % cạnh mỗi thanh bar
    for bar in bars:
        width = bar.get_width()
        plt.text(width + 1, 
                 bar.get_y() + bar.get_height()/2, 
                 f'{width:.1f}%', 
                 va='center', ha='left', fontsize=10, fontweight='bold', color="#264653")
    
    # Tự động căn lề và xử lý layout
    plt.tight_layout()
    
    # Lưu file chuẩn DPI 300
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✅ Hoàn tất! Đã lưu biểu đồ toàn bộ cột rõ nét tại: {save_path}")

# ==========================================
# 3. HÀM THỐNG KÊ BẢNG DỮ LIỆU
# ==========================================
def missing_value_summary(df):
    """
    Trích xuất bảng thống kê chi tiết số lượng và % khuyết thiếu cho toàn bộ cột
    """
    missing_counts = df.isnull().sum()
    missing_percentages = (missing_counts / len(df)) * 100
    
    summary_df = pd.DataFrame({
        'Missing_Count': missing_counts,
        'Missing_Percentage (%)': missing_percentages
    }).sort_values(by='Missing_Percentage (%)', ascending=False)
    
    return summary_df[summary_df['Missing_Count'] > 0]

# ==========================================
# 4. CÁCH GỌI HÀM (THỰC THI CHO TOÀN BỘ CỘT)
# ==========================================
print("Đang làm sạch các giá trị Null ẩn và mảng rỗng trên toàn bộ df_vieclam24h...")
df_v24_cleaned = clean_hidden_nulls(df_vieclam24h)

# Xem bảng thống kê khuyết thiếu của TẤT CẢ CỘT
print("\nBảng thống kê khuyết thiếu (TẤT CẢ CỘT):")
display(missing_value_summary(df_v24_cleaned))

# Vẽ và xuất file ảnh cho tất cả các cột
save_path = r"D:\PYTHON\Data mining\VN-IT-Job-Mining\reports\figures\vieclam24h\01_data_completeness_bar_all.png"
plot_completeness_bar_all(df_v24_cleaned, save_path)

Đang làm sạch các giá trị Null ẩn và mảng rỗng trên toàn bộ df_vieclam24h...

Bảng thống kê khuyết thiếu (TẤT CẢ CỘT):


,Missing_Count,Missing_Percentage (%)
working_time,1195,96.761134
salary,56,4.534413
company_size,12,0.971660
benefits,6,0.485830


✅ Hoàn tất! Đã lưu biểu đồ toàn bộ cột rõ nét tại: D:\PYTHON\Data mining\VN-IT-Job-Mining\reports\figures\vieclam24h\01_data_completeness_bar_all.png


# Trụ Cột 2: Tính Hợp Lệ & Kiểu Dữ Liệu (Validity & Schema Types)

In [14]:
# ==========================================
# 1. HÀM CHẨN ĐOÁN CHUNG VỀ KIỂU DỮ LIỆU (DTYPES)
# ==========================================
def check_data_types(df):
    """
    Kiểm tra kiểu dữ liệu thực tế của các cột và đối chiếu sơ bộ
    """
    print("--- 1. TỔNG QUAN KIỂU DỮ LIỆU ---")
    dtype_summary = pd.DataFrame(df.dtypes, columns=['Thực_tế_Dtype'])
    dtype_summary['Số_lượng_Non_Null'] = df.notnull().sum()
    dtype_summary['Ví_dụ_Giá_trị'] = df.apply(lambda x: x.dropna().iloc[0] if x.notna().any() else np.nan)
    
    # Hiển thị tất cả các cột để dễ nhìn tổng quan sau khi flatten
    display(dtype_summary)

# ==========================================
# 2. HÀM CHẨN ĐOÁN CỘT NGÀY THÁNG (Vieclam24h Format: DD/MM/YYYY)
# ==========================================
def validate_datetime_columns(df):
    """
    Kiểm tra định dạng ngày tháng trên Vieclam24h (Thường là DD/MM/YYYY)
    """
    print("\n--- 2. CHẨN ĐOÁN TÍNH HỢP LỆ NGÀY THÁNG ---")
    # Các cột ngày tháng của Vieclam24h sau khi flatten
    date_cols = ['deadline', 'general_info_posted_date']

    # Regex bắt định dạng DD/MM/YYYY (Ví dụ: 31/10/2026)
    vn_date_pattern = r'^\d{2}/\d{2}/\d{4}$'

    for col in date_cols:
        if col in df.columns:
            valid_values = df[col].dropna().astype(str)
            is_valid = valid_values.str.match(vn_date_pattern)
            
            valid_count = is_valid.sum()
            invalid_count = len(valid_values) - valid_count

            print(f"Cột [{col}]:")
            print(f"  - Đúng định dạng DD/MM/YYYY: {valid_count} bản ghi")
            if invalid_count > 0:
                invalid_samples = valid_values[~is_valid].head(3).tolist()
                print(f"  - ⚠️ SAI ĐỊNH DẠNG: {invalid_count} bản ghi. Ví dụ: {invalid_samples}")
                print("    -> Đề xuất Pre-processing: Cần chuyển đổi thành ISO 8601 (YYYY-MM-DD) để đồng bộ với Warehouse.")
            else:
                print("  - Trạng thái: Hợp lệ 100% theo chuẩn Vieclam24h (Cần convert về ISO khi nạp Lakehouse).")

# ==========================================
# 3. HÀM CHẨN ĐOÁN CỘT SỐ HỌC BỊ STRING HÓA (Đặc sản của Vieclam24h)
# ==========================================
def validate_numeric_strings(df):
    """
    Phát hiện các cột Lương và Kinh nghiệm bị lưu dưới dạng chuỗi chứa text
    """
    print("\n--- 3. CHẨN ĐOÁN SỐ HỌC BỊ STRING HÓA ---")

    # 3.1 Chẩn đoán cột Lương
    if 'salary' in df.columns:
        samples = df['salary'].dropna().head(3).tolist()
        dtype_col = df['salary'].dtype
        print(f"Cột [salary] đang lưu dưới dạng {dtype_col}:")
        print(f"  - Ví dụ giá trị thực tế: {samples}")
        print("  - ⚠️ LỖI NGHIÊM TRỌNG: Lương không phải là biến số (Numeric) mà là một chuỗi (Range String).")
        print("    -> Đề xuất Pre-processing: Phải dùng Regex trích xuất số (ví dụ: '12 - 20 triệu' -> min=12, max=20).")

    # 3.2 Chẩn đoán cột Kinh nghiệm
    if 'experience_summary' in df.columns:
        samples = df['experience_summary'].dropna().head(3).tolist()
        dtype_col = df['experience_summary'].dtype
        print(f"\nCột [experience_summary] đang lưu dưới dạng {dtype_col}:")
        print(f"  - Ví dụ giá trị thực tế: {samples}")
        print("  - ⚠️ LỖI: Chứa ký tự 'năm'. Không thể dùng để vẽ Scatter Plot ngay lập tức.")
        print("    -> Đề xuất Pre-processing: Loại bỏ chữ 'năm', chuyển 'Chưa có kinh nghiệm' thành 0, sau đó ép kiểu Int.")

# ==========================================
# 4. HÀM CHẨN ĐOÁN CẤU TRÚC MẢNG (LIST)
# ==========================================
def validate_complex_structures(df):
    """
    Kiểm tra các cột chứa cấu trúc List xem có phải là list chuỗi phẳng (flat list) không.
    """
    print("\n--- 4. CHẨN ĐOÁN CẤU TRÚC MẢNG (VĂN BẢN VÀ KỸ NĂNG) ---")
    array_cols = ['description', 'requirements', 'benefits', 'keywords', 'general_info_workplaces']

    for col in array_cols:
        if col in df.columns:
            sample_val = df[col].dropna().iloc[0]
            actual_type = type(sample_val)

            print(f"Cột [{col}]:")
            if actual_type == list:
                # Kiểm tra cấu trúc bên trong list
                inner_type = type(sample_val[0]) if len(sample_val) > 0 else "Empty"
                print(f"  - Bề mặt: List")
                print(f"  - Cấu trúc bên trong: List of {inner_type}")
                if inner_type == str:
                    print("  - Trạng thái: Rất tốt. Đây là mảng phẳng (flat list of strings), dễ dàng xử lý NLP sau này.")
                elif inner_type == dict:
                    print("  - ⚠️ Cảnh báo: Chứa JSON lồng nhau, cần trích xuất khóa.")
            elif actual_type == str:
                print(f"  - ⚠️ LỖI: Mảng đã bị string hóa thành {actual_type}. Cần dùng ast.literal_eval().")

# ==========================================
# THỰC THI PIPELINE CHẨN ĐOÁN TRỤ CỘT 2
# ==========================================
print("BẮT ĐẦU CHẨN ĐOÁN TRỤ CỘT 2: TÍNH HỢP LỆ DỮ LIỆU (VIECLAM24H)\n" + "="*60)

# Giả định df_vieclam24h là DataFrame bạn đã flatten từ bước đọc file
check_data_types(df_vieclam24h)
validate_datetime_columns(df_vieclam24h)
validate_numeric_strings(df_vieclam24h)
validate_complex_structures(df_vieclam24h)

BẮT ĐẦU CHẨN ĐOÁN TRỤ CỘT 2: TÍNH HỢP LỆ DỮ LIỆU (VIECLAM24H)
--- 1. TỔNG QUAN KIỂU DỮ LIỆU ---


,Thực_tế_Dtype,Số_lượng_Non_Null,Ví_dụ_Giá_trị
title,object,1235,Nhân Viên Kinh Doanh - Tư Vấn Giải Pháp Lương ...
salary,object,1235,12 - 20 triệu
deadline,object,1235,31/10/2026
location_summary,object,1235,Hà Nội
experience_summary,object,1235,1 năm
description,object,1235,"[Tìm kiếm, phát triển và mở rộng tệp khách hàn..."
requirements,object,1235,[Tốt nghiệp Cao đẳng/Đại học các ngành Kinh tế...
benefits,object,1235,[Mức lương cơ bản:12.000.000 - 15.000.000 + ph...
working_time,object,1235,
working_address,object,1235,"Tầng 24, tòa MHDI, 68 Nguyễn Cơ Thạch, Mỹ Đình..."



--- 2. CHẨN ĐOÁN TÍNH HỢP LỆ NGÀY THÁNG ---
Cột [deadline]:
  - Đúng định dạng DD/MM/YYYY: 1235 bản ghi
  - Trạng thái: Hợp lệ 100% theo chuẩn Vieclam24h (Cần convert về ISO khi nạp Lakehouse).
Cột [general_info_posted_date]:
  - Đúng định dạng DD/MM/YYYY: 1235 bản ghi
  - Trạng thái: Hợp lệ 100% theo chuẩn Vieclam24h (Cần convert về ISO khi nạp Lakehouse).

--- 3. CHẨN ĐOÁN SỐ HỌC BỊ STRING HÓA ---
Cột [salary] đang lưu dưới dạng object:
  - Ví dụ giá trị thực tế: ['12 - 20 triệu', '8 - 12 triệu', '20 - 25 triệu']
  - ⚠️ LỖI NGHIÊM TRỌNG: Lương không phải là biến số (Numeric) mà là một chuỗi (Range String).
    -> Đề xuất Pre-processing: Phải dùng Regex trích xuất số (ví dụ: '12 - 20 triệu' -> min=12, max=20).

Cột [experience_summary] đang lưu dưới dạng object:
  - Ví dụ giá trị thực tế: ['1 năm', '2 năm', '2 năm']
  - ⚠️ LỖI: Chứa ký tự 'năm'. Không thể dùng để vẽ Scatter Plot ngay lập tức.
    -> Đề xuất Pre-processing: Loại bỏ chữ 'năm', chuyển 'Chưa có kinh nghiệm' thành 0, sau 

# Biểu đồ 2:

In [15]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import re

# ==========================================
# 1. HÀM TIỀN XỬ LÝ (TRÍCH XUẤT LƯƠNG TỪ CHUỖI)
# ==========================================
def extract_max_salary(salary_str):
    """
    Bóc tách mức lương tối đa từ chuỗi văn bản (đơn vị: Triệu VNĐ).
    Xử lý các case: "12 - 20 triệu", "Lên đến 30 triệu", "Thỏa thuận", "20.000.000 VNĐ"
    """
    if pd.isna(salary_str):
        return np.nan
        
    val = str(salary_str).lower().strip()
    
    # Bỏ qua các tin không có số liệu cụ thể
    if 'thỏa thuận' in val or 'thương lượng' in val or 'cạnh tranh' in val:
        return np.nan
        
    # Thay dấu phẩy thành chấm để xử lý số thập phân (ví dụ 12,5 triệu -> 12.5)
    val = val.replace(',', '.')
    
    # Tìm tất cả các con số trong chuỗi (bao gồm cả số thập phân)
    nums = re.findall(r'\d+\.?\d*', val)
    
    if not nums:
        return np.nan
        
    # Ép kiểu sang float và lấy giá trị lớn nhất (Salary Max)
    nums = [float(n) for n in nums]
    max_val = max(nums)
    
    # Quy đổi về mốc Triệu VNĐ 
    # (Nếu HR nhập 20000000 thay vì 20 triệu, ta chia cho 1 triệu)
    if max_val >= 1000:
        max_val = max_val / 1_000_000
        
    # Loại bỏ các ngoại lai do gõ nhầm (ví dụ: lương > 500 triệu/tháng hoặc = 0)
    if max_val <= 0 or max_val > 500:
        return np.nan
        
    return max_val

# ==========================================
# 2. HÀM VẼ BIỂU ĐỒ PHÂN PHỐI LƯƠNG
# ==========================================
def plot_salary_distribution_v24(df, save_path):
    """
    Vẽ 2 biểu đồ: Phân phối lương gốc (Triệu VNĐ) và Phân phối sau khi biến đổi Log.
    Tính toán Skewness, Mean, Median cho dữ liệu Vieclam24h.
    """
    if 'salary' not in df.columns:
        print("❌ Không tìm thấy cột 'salary' trong DataFrame.")
        return
        
    # 1. Trích xuất dữ liệu lương thực tế
    print("Đang bóc tách số liệu từ cột chuỗi lương...")
    salaries_millions = df['salary'].apply(extract_max_salary).dropna()
    
    if salaries_millions.empty:
        print("❌ Không có dữ liệu lương hợp lệ để vẽ biểu đồ.")
        return
        
    # 2. Khởi tạo không gian vẽ
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # ==========================================
    # BIỂU ĐỒ TRÁI: PHÂN PHỐI LƯƠNG GỐC
    # ==========================================
    sns.histplot(salaries_millions, kde=True, ax=axes[0], color="#1d3557", bins=30)
    
    median_val = salaries_millions.median()
    mean_val = salaries_millions.mean()
    skewness_val = salaries_millions.skew()
    
    axes[0].axvline(median_val, color="red", linestyle="--", linewidth=2, label=f"Median: {median_val:.1f}M")
    axes[0].axvline(mean_val, color="orange", linestyle="-", linewidth=2, label=f"Mean: {mean_val:.1f}M")
    
    axes[0].set_title(f"Phân Phối Lương Gốc (Skewness = {skewness_val:.2f})", fontsize=14, pad=15)
    axes[0].set_xlabel("Triệu VNĐ/tháng", fontsize=12)
    axes[0].set_ylabel("Số lượng tin tuyển dụng", fontsize=12)
    axes[0].legend(fontsize=11)
    
    # ==========================================
    # BIỂU ĐỒ PHẢI: LOG-TRANSFORMED
    # ==========================================
    log_salaries = np.log1p(salaries_millions)
    log_skewness = log_salaries.skew()
    
    sns.histplot(log_salaries, kde=True, ax=axes[1], color="#e76f51", bins=30)
    
    axes[1].set_title(f"Phân Phối Sau Log-Transform (Skewness = {log_skewness:.2f})", fontsize=14, pad=15)
    axes[1].set_xlabel("log(1 + Lương [Triệu VNĐ])", fontsize=12)
    axes[1].set_ylabel("Số lượng tin tuyển dụng", fontsize=12)
    
    # 3. Căn chỉnh và Lưu file
    plt.tight_layout()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✅ Hoàn tất! Đã lưu biểu đồ phân phối lương tại: {save_path}")

# ==========================================
# THỰC THI PIPELINE VẼ BIỂU ĐỒ 2 (VIECLAM24H)
# ==========================================
# Giả định dữ liệu nằm trong df_vieclam24h
save_path_chart_2 = r"D:\PYTHON\Data mining\VN-IT-Job-Mining\reports\figures\vieclam24h\02_salary_distribution_skewness.png"
plot_salary_distribution_v24(df_vieclam24h, save_path_chart_2)

Đang bóc tách số liệu từ cột chuỗi lương...
✅ Hoàn tất! Đã lưu biểu đồ phân phối lương tại: D:\PYTHON\Data mining\VN-IT-Job-Mining\reports\figures\vieclam24h\02_salary_distribution_skewness.png


# Trụ cột 3: Tính Duy Nhất & Trùng Lặp (Uniqueness & Duplication)

In [16]:
def check_duplication_v24(df):
    """
    Hàm đo lường tỷ lệ trùng lặp dữ liệu theo 3 cấp độ cho Vieclam24h:
    1. Trùng lặp tuyệt đối (Exact duplicates)
    2. Trùng lặp khóa định danh (ID / URL duplicates)
    3. Trùng lặp logic (Semantic duplicates)
    """
    print("--- TRỤ CỘT 3: CHẨN ĐOÁN TÍNH DUY NHẤT & TRÙNG LẶP (VIECLAM24H) ---")
    total_records = len(df)
    print(f"Tổng số bản ghi ban đầu: {total_records}")

    if total_records == 0:
        print("❌ DataFrame rỗng, không thể kiểm tra trùng lặp.")
        return

    # ==========================================
    # 1. TRÙNG LẶP TUYỆT ĐỐI (100% CÁC CỘT GIỐNG NHAU)
    # ==========================================
    # Ép toàn bộ DataFrame về dạng chuỗi (string) để tránh lỗi unhashable type do chứa list
    df_string_format = df.astype(str)
    exact_dups = df_string_format.duplicated().sum()
    exact_pct = (exact_dups / total_records) * 100
    
    print(f"\n1. Trùng lặp tuyệt đối (Exact Duplicates):")
    print(f"   - Số lượng: {exact_dups} bản ghi")
    print(f"   - Tỷ lệ: {exact_pct:.2f}%")

    # ==========================================
    # 2. TRÙNG LẶP KHÓA ĐỊNH DANH (ID / URL)
    # ==========================================
    # Kiểm tra cột ID hoặc URL của Vieclam24h
    # (Tùy thuộc vào việc bạn có flatten 'metadata' hay không. Nếu chỉ flatten 'raw_payload', 
    # Vieclam24h không có sẵn ID trong raw_payload. Ta dùng 'title' + 'company_name' làm fallback nếu thiếu metadata)
    
    id_col = None
    if 'metadata_job_id' in df.columns:
        id_col = 'metadata_job_id'
    elif 'metadata_source_url' in df.columns:
        id_col = 'metadata_source_url'

    if id_col:
        id_dups = df.duplicated(subset=[id_col]).sum()
        id_pct = (id_dups / total_records) * 100
        print(f"\n2. Trùng lặp khóa định danh (Cột '{id_col}'):")
        print(f"   - Số lượng: {id_dups} bản ghi")
        print(f"   - Tỷ lệ: {id_pct:.2f}%")
    else:
        id_dups = 0
        print("\n2. Trùng lặp khóa định danh: Không tìm thấy cột chứa Job ID hay URL trong DataFrame hiện tại.")
        print("   -> Ghi chú: Có thể bạn đã bỏ qua node 'metadata' khi đọc JSON.")

    # ==========================================
    # 3. TRÙNG LẶP LOGIC (COMPANY + TITLE + LOCATION)
    # ==========================================
    # Các cột cốt lõi của Vieclam24h
    logical_cols = ['title', 'company_name', 'location_summary']
    existing_logical_cols = [c for c in logical_cols if c in df.columns]
    
    if len(existing_logical_cols) == len(logical_cols):
        # Chuẩn hóa (lower-case & strip) để phát hiện HR đăng lại tin nhưng gõ khác in hoa/in thường
        temp_df = df[logical_cols].fillna('').astype(str).apply(lambda x: x.str.lower().str.strip())
        logical_dups = temp_df.duplicated().sum()
        logical_pct = (logical_dups / total_records) * 100
        
        print(f"\n3. Trùng lặp logic (Theo {' + '.join(logical_cols)}):")
        print(f"   - Số lượng: {logical_dups} bản ghi")
        print(f"   - Tỷ lệ: {logical_pct:.2f}%")
    else:
        logical_dups = 0
        print(f"\n3. Trùng lặp logic: Bỏ qua do thiếu một trong các cột: {logical_cols}.")

    # ==========================================
    # KẾT LUẬN & ĐỀ XUẤT PRE-PROCESSING
    # ==========================================
    print("\n" + "="*50)
    max_dups = max(exact_dups, id_dups, logical_dups)
    if max_dups > 0:
        print("⚠️ CHẨN ĐOÁN: Dữ liệu Vieclam24h chứa bản ghi trùng lặp (Overlapping).")
        print("💡 HƯỚNG XỬ LÝ (Pre-processing):")
        if id_col:
            print(f"   -> Bước 1: Gọi drop_duplicates(subset=['{id_col}']) để xóa tin cào trùng do crawler bị loop phân trang.")
        print(f"   -> Bước 2: Gọi drop_duplicates(subset={logical_cols}) và giữ lại tin mới nhất (dựa trên 'general_info_posted_date') để xóa tin bị HR đăng lặp (Repost).")
    else:
        print("✅ TUYỆT VỜI: Không phát hiện bản ghi trùng lặp nào. Pipeline collection hoạt động hoàn hảo.")

# ==========================================
# THỰC THI PIPELINE CHẨN ĐOÁN TRÙNG LẶP
# ==========================================
# Giả định dữ liệu nằm trong df_vieclam24h
check_duplication_v24(df_vieclam24h)

--- TRỤ CỘT 3: CHẨN ĐOÁN TÍNH DUY NHẤT & TRÙNG LẶP (VIECLAM24H) ---
Tổng số bản ghi ban đầu: 1235

1. Trùng lặp tuyệt đối (Exact Duplicates):
   - Số lượng: 616 bản ghi
   - Tỷ lệ: 49.88%

2. Trùng lặp khóa định danh (Cột 'metadata_job_id'):
   - Số lượng: 616 bản ghi
   - Tỷ lệ: 49.88%

3. Trùng lặp logic (Theo title + company_name + location_summary):
   - Số lượng: 619 bản ghi
   - Tỷ lệ: 50.12%

⚠️ CHẨN ĐOÁN: Dữ liệu Vieclam24h chứa bản ghi trùng lặp (Overlapping).
💡 HƯỚNG XỬ LÝ (Pre-processing):
   -> Bước 1: Gọi drop_duplicates(subset=['metadata_job_id']) để xóa tin cào trùng do crawler bị loop phân trang.
   -> Bước 2: Gọi drop_duplicates(subset=['title', 'company_name', 'location_summary']) và giữ lại tin mới nhất (dựa trên 'general_info_posted_date') để xóa tin bị HR đăng lặp (Repost).


# Trụ Cột 5: Độ Thiên Lệch & Tính Mất Cân Bằng (Bias & Balance Profiling)

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ==========================================
# 1. HÀM ĐÁNH GIÁ CHỈ SỐ MẤT CÂN BẰNG (TEXT REPORT)
# ==========================================
def profile_bias_and_balance_v24(df):
    print("--- TRỤ CỘT 5: CHẨN ĐOÁN THIÊN LỆCH & MẤT CÂN BẰNG (VIECLAM24H) ---")
    total_records = len(df)
    if total_records == 0:
        return
        
    # 1. Mất cân bằng Cấp bậc (Seniority Bias)
    # Vieclam24h lưu cấp bậc trong 'general_info_level'
    level_col = 'general_info_level'
    if level_col in df.columns:
        print("\n1. Phân bổ Cấp bậc (Seniority):")
        seniority_counts = df[level_col].value_counts()
        for level, count in seniority_counts.items():
            pct = (count / total_records) * 100
            print(f"   - {level}: {count} tin ({pct:.1f}%)")
        if len(seniority_counts) >= 2:
            imbalance_ratio = seniority_counts.max() / seniority_counts.min()
            print(f"   => Imbalance Ratio (Max/Min): {imbalance_ratio:.1f}x")

    # 2. Thiên lệch Địa lý (Geographic Bias)
    # Vieclam24h có 'location_summary' khá sạch
    loc_col = 'location_summary'
    if loc_col in df.columns:
        print("\n2. Phân bổ Địa lý (Location):")
        geo_counts = df[loc_col].value_counts().head(5)
        for loc, count in geo_counts.items():
            pct = (count / total_records) * 100
            print(f"   - {loc}: {count} tin ({pct:.1f}%)")

    # 3. Tính Thưa thớt Kỹ năng (Skill Sparsity - Pareto 80/20)
    if 'keywords' in df.columns:
        # Bóc tách mảng kỹ năng thành các dòng đơn lẻ
        all_skills = df['keywords'].dropna().explode().astype(str).str.strip()
        
        # BỘ LỌC RÁC VIECLAM24H: Loại bỏ các keywords quá dài (chứa mô tả công việc hoặc tên công ty)
        # VD: "Công Ty Cổ Phần Công ...", "nhân viên kinh doanh thị trường"
        valid_skills = all_skills[all_skills.str.len() <= 20]
        
        skill_counts = valid_skills.value_counts()
        total_skill_tags = len(valid_skills)
        unique_skills = len(skill_counts)
        
        # Tính Pareto
        if total_skill_tags > 0:
            cumulative_pct = (skill_counts.cumsum() / total_skill_tags) * 100
            top_skills_for_80_pct = cumulative_pct[cumulative_pct <= 80].count()
            pareto_ratio = (top_skills_for_80_pct / unique_skills) * 100
            
            print("\n3. Chẩn đoán Kỹ năng (Quy luật Pareto) - Đã lọc rác:")
            print(f"   - Tổng số tags hợp lệ xuất hiện: {total_skill_tags}")
            print(f"   - Số lượng kỹ năng duy nhất (Unique): {unique_skills}")
            print(f"   - Nhóm Top {top_skills_for_80_pct} kỹ năng (chiếm {pareto_ratio:.1f}% unique skills) đang thống trị 80% nhu cầu thị trường.")
            print(f"   - ⚠️ Chẩn đoán: Dữ liệu keywords của Vieclam24h chứa rất nhiều nhiễu (spam SEO). Việc lọc rác bằng Regex là bắt buộc ở giai đoạn Pre-processing.")

# ==========================================
# 2. HÀM VẼ BIỂU ĐỒ 6: PARETO 80/20 KỸ NĂNG
# ==========================================
def plot_skills_pareto_v24(df, save_path):
    if 'keywords' not in df.columns:
        return
        
    all_skills = df['keywords'].dropna().explode().astype(str).str.strip()
    
    # Áp dụng bộ lọc tương tự như trên để biểu đồ không bị vỡ layout
    valid_skills = all_skills[all_skills.str.len() <= 20]
    
    top20 = valid_skills.value_counts().head(20).reset_index()
    top20.columns = ["Skill", "Frequency"]
    
    if top20.empty:
        return
        
    top20["Cum_Percent"] = (top20["Frequency"].cumsum() / top20["Frequency"].sum()) * 100
    
    fig, ax1 = plt.subplots(figsize=(13, 6))
    ax2 = ax1.twinx()
    
    # Vẽ Bar Chart tần suất
    sns.barplot(data=top20, x="Skill", y="Frequency", ax=ax1, palette="viridis", hue="Skill", legend=False)
    ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha="right", fontsize=10)
    ax1.set_ylabel("Tần suất xuất hiện", fontsize=11)
    
    # Vẽ đường Tích lũy (Cumulative Line)
    sns.lineplot(data=top20, x="Skill", y="Cum_Percent", ax=ax2, color="#d62828", marker="o", linewidth=2.5)
    ax2.axhline(80, color="gray", linestyle="--", alpha=0.7, label="Ngưỡng 80% Pareto")
    ax2.set_ylabel("Tỷ lệ tích lũy (%)", fontsize=11)
    ax2.set_ylim(0, 105)
    ax2.legend(loc="lower right")
    
    plt.title("Phân Phối Tần Suất Kỹ Năng & Pareto (Vieclam24h)", fontsize=14, fontweight='bold', pad=15)
    plt.tight_layout()
    
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Đã lưu Biểu đồ 6 (Pareto Skills) tại: {save_path}")

# ==========================================
# 3. HÀM VẼ BIỂU ĐỒ 7: CHẨN ĐOÁN MẤT CÂN BẰNG LỚP
# ==========================================
def plot_bias_and_balance_v24(df, save_path):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Biểu đồ 1: Mất cân bằng Cấp bậc (Seniority Class Imbalance)
    level_col = 'general_info_level'
    if level_col in df.columns:
        seniority_counts = df[level_col].value_counts()
        sns.barplot(x=seniority_counts.values, y=seniority_counts.index, ax=axes[0], palette="flare", hue=seniority_counts.index, legend=False)
        
        imbalance_ratio = seniority_counts.max() / seniority_counts.min() if len(seniority_counts) >= 2 else 1
        axes[0].set_title(f"Mất Cân Bằng Cấp Bậc\n(Imbalance Ratio = {imbalance_ratio:.1f}x)", fontsize=13, pad=10)
        axes[0].set_xlabel("Số lượng tin tuyển dụng", fontsize=11)
        axes[0].set_ylabel("")
    
    # Biểu đồ 2: Thiên lệch Địa lý (Donut Chart)
    loc_col = 'location_summary'
    if loc_col in df.columns:
        top_loc = df[loc_col].value_counts().head(5)
        # Gộp các tỉnh còn lại thành "Khác"
        if len(df[loc_col].value_counts()) > 5:
            other_count = df[loc_col].value_counts().iloc[5:].sum()
            top_loc['Các tỉnh khác'] = other_count
            
        axes[1].pie(top_loc, labels=top_loc.index, autopct="%1.1f%%", startangle=140,
                    colors=sns.color_palette("Set2"), pctdistance=0.8, 
                    textprops={'fontsize': 11})
        # Biến Pie thành Donut
        centre_circle = plt.Circle((0,0), 0.55, fc='white')
        axes[1].add_artist(centre_circle)
        axes[1].set_title("Thiên Lệch Địa Lý (Geographic Bias)", fontsize=13, pad=10)
    
    plt.tight_layout()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Đã lưu Biểu đồ 7 (Bias & Balance) tại: {save_path}")

# ==========================================
# THỰC THI PIPELINE 
# ==========================================
# 1. In báo cáo Text ra console
profile_bias_and_balance_v24(df_vieclam24h)

# 2. Vẽ và xuất biểu đồ
path_chart_6 = r"D:\PYTHON\Data mining\VN-IT-Job-Mining\reports\figures\vieclam24h\03_skills_pareto_distribution.png"
path_chart_7 = r"D:\PYTHON\Data mining\VN-IT-Job-Mining\reports\figures\vieclam24h\04_bias_and_class_balance.png"

plot_skills_pareto_v24(df_vieclam24h, path_chart_6)
plot_bias_and_balance_v24(df_vieclam24h, path_chart_7)

--- TRỤ CỘT 5: CHẨN ĐOÁN THIÊN LỆCH & MẤT CÂN BẰNG (VIECLAM24H) ---

1. Phân bổ Cấp bậc (Seniority):
   - Nhân viên: 1111 tin (90.0%)
   - Cộng tác viên: 42 tin (3.4%)
   - Trưởng phòng: 30 tin (2.4%)
   - Chuyên gia: 26 tin (2.1%)
   - Trưởng nhóm: 24 tin (1.9%)
   - Giám đốc: 2 tin (0.2%)
   => Imbalance Ratio (Max/Min): 555.5x

2. Phân bổ Địa lý (Location):
   - TP.HCM: 498 tin (40.3%)
   - Hà Nội: 345 tin (27.9%)
   - Đà Nẵng: 62 tin (5.0%)
   - Tây Ninh: 32 tin (2.6%)
   - Vĩnh Long: 28 tin (2.3%)

3. Chẩn đoán Kỹ năng (Quy luật Pareto) - Đã lọc rác:
   - Tổng số tags hợp lệ xuất hiện: 8598
   - Số lượng kỹ năng duy nhất (Unique): 921
   - Nhóm Top 318 kỹ năng (chiếm 34.5% unique skills) đang thống trị 80% nhu cầu thị trường.
   - ⚠️ Chẩn đoán: Dữ liệu keywords của Vieclam24h chứa rất nhiều nhiễu (spam SEO). Việc lọc rác bằng Regex là bắt buộc ở giai đoạn Pre-processing.


C:\Users\LAPTOP\AppData\Local\Temp\ipykernel_10256\2425476889.py:89: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha="right", fontsize=10)


✅ Đã lưu Biểu đồ 6 (Pareto Skills) tại: D:\PYTHON\Data mining\VN-IT-Job-Mining\reports\figures\vieclam24h\03_skills_pareto_distribution.png
✅ Đã lưu Biểu đồ 7 (Bias & Balance) tại: D:\PYTHON\Data mining\VN-IT-Job-Mining\reports\figures\vieclam24h\04_bias_and_class_balance.png


# Headmap

In [18]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import re

# ==========================================
# 1. HÀM TIỀN XỬ LÝ: TRÍCH XUẤT ĐẶC TRƯNG SỐ HỌC
# ==========================================
def extract_salary_min_max(salary_str):
    """
    Bóc tách lương Min và Max từ chuỗi (ví dụ: '12 - 20 triệu' -> min=12, max=20)
    """
    if pd.isna(salary_str) or 'thỏa thuận' in str(salary_str).lower():
        return np.nan, np.nan
        
    s = str(salary_str).lower().replace(',', '.')
    nums = [float(n) for n in re.findall(r'\d+\.?\d*', s)]
    
    if not nums:
        return np.nan, np.nan
        
    # Xử lý quy đổi về Triệu VNĐ nếu HR nhập số lớn (ví dụ: 15000000)
    nums = [n / 1_000_000 if n >= 1000 else n for n in nums]
    
    if len(nums) >= 2:
        return min(nums), max(nums)
    elif len(nums) == 1:
        if 'lên đến' in s or 'tới' in s:
            return np.nan, nums[0]
        elif 'từ' in s or 'trên' in s:
            return nums[0], np.nan
        return nums[0], nums[0]
    return np.nan, np.nan

def extract_experience_years(exp_str):
    """
    Bóc tách số năm kinh nghiệm từ chuỗi (ví dụ: '1 năm' -> 1)
    """
    if pd.isna(exp_str):
        return np.nan
    s = str(exp_str).lower()
    if 'chưa' in s or 'không' in s:
        return 0.0
    nums = re.findall(r'\d+', s)
    return float(nums[0]) if nums else np.nan

# ==========================================
# 2. HÀM VẼ MA TRẬN TƯƠNG QUAN
# ==========================================
def plot_correlation_heatmap_v24(df, save_path):
    print("--- VẼ MA TRẬN TƯƠNG QUAN HẠNG SPEARMAN (VIECLAM24H) ---")
    
    # 1. KHỞI TẠO DATAFRAME CHỨA BIẾN SỐ HỌC
    df_num = pd.DataFrame()
    
    # Trích xuất Lương
    if 'salary' in df.columns:
        print(" Đang trích xuất Lương Min/Max...")
        salary_tuples = df['salary'].apply(extract_salary_min_max)
        df_num['Salary_Min_M'] = salary_tuples.apply(lambda x: x[0])
        df_num['Salary_Max_M'] = salary_tuples.apply(lambda x: x[1])
        
    # Trích xuất Kinh nghiệm
    if 'experience_summary' in df.columns:
        print(" Đang trích xuất Số năm kinh nghiệm...")
        df_num['Years_of_Experience'] = df['experience_summary'].apply(extract_experience_years)
        
    # Tạo biến đếm số lượng Kỹ năng (Keywords)
    if 'keywords' in df.columns:
        print(" Đang đếm số lượng Tags kỹ năng...")
        df_num['Total_Skills_Count'] = df['keywords'].apply(
            lambda x: len(x) if isinstance(x, list) else 0
        )
        
    # Tạo biến đếm độ dài Mô tả công việc (Word count của description + requirements)
    print(" Đang tính toán độ dài văn bản JD...")
    def count_words(lst):
        if not isinstance(lst, list): return 0
        return sum(len(str(item).split()) for item in lst)
        
    jd_desc = df['description'].apply(count_words) if 'description' in df.columns else 0
    jd_req = df['requirements'].apply(count_words) if 'requirements' in df.columns else 0
    df_num['JD_Word_Count'] = jd_desc + jd_req

    # 2. TÍNH TOÁN MA TRẬN TƯƠNG QUAN SPEARMAN
    # Dùng Spearman để chịu lỗi tốt hơn với dữ liệu bị lệch (outliers)
    corr_matrix = df_num.corr(method="spearman")
    
    if corr_matrix.isna().all().all():
        print("❌ Không đủ dữ liệu định lượng hợp lệ để vẽ Heatmap.")
        return None
        
    # 3. VẼ HEATMAP TRỰC QUAN
    plt.figure(figsize=(9, 7))
    
    # Mask ẩn nửa tam giác trên
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    
    sns.heatmap(
        corr_matrix, 
        mask=mask, 
        annot=True,          
        fmt=".2f",           
        cmap="coolwarm",     
        vmin=-1, vmax=1,     
        linewidths=1.5, 
        cbar_kws={"shrink": 0.8},
        annot_kws={"size": 11, "weight": "bold"}
    )
    
    plt.title("Ma Trận Tương Quan Hạng Spearman\n(Các Thuộc Tính Định Lượng - Vieclam24h)", 
              fontsize=14, fontweight='bold', pad=20)
    
    plt.xticks(rotation=45, ha='right', fontsize=11)
    plt.yticks(rotation=0, fontsize=11)
    plt.tight_layout()
    
    # 4. LƯU FILE
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✅ Hoàn tất! Đã lưu Heatmap tương quan tại: {save_path}")
    return corr_matrix

# ==========================================
# THỰC THI PIPELINE VẼ BIỂU ĐỒ 5
# ==========================================
# Giả định dữ liệu nằm trong df_vieclam24h
save_path_chart_5 = r"D:\PYTHON\Data mining\VN-IT-Job-Mining\reports\figures\vieclam24h\05_spearman_correlation_matrix.png"

correlation_table = plot_correlation_heatmap_v24(df_vieclam24h, save_path_chart_5)
display(correlation_table)

--- VẼ MA TRẬN TƯƠNG QUAN HẠNG SPEARMAN (VIECLAM24H) ---
 Đang trích xuất Lương Min/Max...
 Đang trích xuất Số năm kinh nghiệm...
 Đang đếm số lượng Tags kỹ năng...
 Đang tính toán độ dài văn bản JD...
✅ Hoàn tất! Đã lưu Heatmap tương quan tại: D:\PYTHON\Data mining\VN-IT-Job-Mining\reports\figures\vieclam24h\05_spearman_correlation_matrix.png


,Salary_Min_M,Salary_Max_M,Years_of_Experience,Total_Skills_Count,JD_Word_Count
Salary_Min_M,1.000000,0.587523,0.489160,0.003789,0.323045
Salary_Max_M,0.587523,1.000000,0.279198,0.013487,0.152563
Years_of_Experience,0.489160,0.279198,1.000000,-0.088934,0.390654
Total_Skills_Count,0.003789,0.013487,-0.088934,1.000000,0.025732
JD_Word_Count,0.323045,0.152563,0.390654,0.025732,1.000000
